# Latent Space Diffusion Model Training

This notebook implements the **diffusion model** in the latent space of functional data.

## Architecture Overview

1. Load the pre-trained **Encoder** from Phase 1
2. Encode all training functional data into latent vectors `z ∈ R^d`
3. Train a **Score-Based Diffusion Model (SDE)** in the low-dimensional latent space
4. Save the trained diffusion model for generation

This approach is computationally efficient because:
- Diffusion operates in low-dimensional space (e.g., d=64) rather than high-dimensional functional space (e.g., 100+ points)
- Training and sampling are orders of magnitude faster
- Achieves minimax-optimal density estimation for functional data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import os

# Import scikit-fda
from skfda.representation.grid import FDataGrid

## 1. Load Pre-trained Encoder

In [ ]:
# Import encoder architecture from previous notebook
class FunctionalEncoder(nn.Module):
    """Encoder network for functional data."""
    
    def __init__(self, latent_dim=64, n_features=1):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.conv1 = nn.Conv1d(n_features, 32, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(128)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, latent_dim)
    
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.global_pool(x)
        x = x.squeeze(-1)
        z = self.fc(x)
        return z

# Load the trained encoder
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

checkpoint = torch.load('models/functional_encoder.pth', map_location=device)
latent_dim = checkpoint['latent_dim']
n_features = checkpoint['n_features']

encoder = FunctionalEncoder(latent_dim=latent_dim, n_features=n_features)
encoder.load_state_dict(checkpoint['encoder_state_dict'])
encoder = encoder.to(device)
encoder.eval()

print(f"Encoder loaded successfully!")
print(f"Latent dimension: {latent_dim}")

## 2. Generate and Encode Training Data

In [ ]:
def generate_toy_functional_data(n_samples=1000, n_points=100, domain_range=(0, 1)):
    """Generate synthetic functional data (same as in notebook 1)."""
    t = np.linspace(domain_range[0], domain_range[1], n_points)
    data_matrix = np.zeros((n_samples, n_points, 1))
    
    for i in range(n_samples):
        A = np.random.uniform(0.5, 2.0)
        omega = np.random.uniform(2 * np.pi, 6 * np.pi)
        phi = np.random.uniform(0, 2 * np.pi)
        noise = np.random.normal(0, 0.05, n_points)
        x_t = A * np.sin(omega * t + phi) + noise
        data_matrix[i, :, 0] = x_t
    
    return FDataGrid(data_matrix=data_matrix, grid_points=t)

# Generate training data
print("Generating functional data for encoding...")
n_train = 2000  # More samples for better diffusion training
fdata_train = generate_toy_functional_data(n_samples=n_train, n_points=100)
print(f"Generated {n_train} functional samples")

In [ ]:
# Encode all training data into latent space
print("Encoding functional data into latent space...")

data_tensor = torch.FloatTensor(fdata_train.data_matrix).to(device)
latent_codes = []

batch_size = 128
with torch.no_grad():
    for i in range(0, len(data_tensor), batch_size):
        batch = data_tensor[i:i+batch_size]
        z = encoder(batch)
        latent_codes.append(z.cpu())

# Concatenate all latent codes
latent_codes = torch.cat(latent_codes, dim=0)
print(f"Latent codes shape: {latent_codes.shape}")

# Visualize latent space distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot first two dimensions
axes[0].scatter(latent_codes[:, 0].numpy(), latent_codes[:, 1].numpy(), alpha=0.5, s=10)
axes[0].set_xlabel('Latent Dimension 1')
axes[0].set_ylabel('Latent Dimension 2')
axes[0].set_title('Latent Space (2D projection)')
axes[0].grid(True, alpha=0.3)

# Plot distribution of latent values
axes[1].hist(latent_codes.numpy().flatten(), bins=50, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Latent Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Latent Values')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLatent code statistics:")
print(f"  Mean: {latent_codes.mean():.4f}")
print(f"  Std: {latent_codes.std():.4f}")
print(f"  Min: {latent_codes.min():.4f}")
print(f"  Max: {latent_codes.max():.4f}")

## 3. Define the Diffusion Model

We implement a simple **DDPM (Denoising Diffusion Probabilistic Model)** for the latent space.

The key idea:
- Forward process: Gradually add Gaussian noise to latent codes
- Reverse process: Train a neural network to predict and remove noise
- Generation: Start from pure noise and iteratively denoise to create new latent codes

In [ ]:
class NoiseSchedule:
    """Linear noise schedule for diffusion."""
    
    def __init__(self, n_steps=1000, beta_start=1e-4, beta_end=0.02):
        self.n_steps = n_steps
        
        # Linear schedule for beta
        self.betas = torch.linspace(beta_start, beta_end, n_steps)
        
        # Precompute useful quantities
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion q(x_t | x_0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        
        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
    
    def q_sample(self, x_0, t, noise=None):
        """Sample from q(x_t | x_0) - the forward diffusion process."""
        if noise is None:
            noise = torch.randn_like(x_0)
        
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t].reshape(-1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1)
        
        return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise

# Test noise schedule
noise_schedule = NoiseSchedule(n_steps=1000)
print(f"Noise schedule created with {noise_schedule.n_steps} steps")
print(f"Beta range: [{noise_schedule.betas[0]:.6f}, {noise_schedule.betas[-1]:.6f}]")

In [ ]:
class LatentDiffusionModel(nn.Module):
    """Simple MLP-based noise prediction network for latent diffusion."""
    
    def __init__(self, latent_dim=64, hidden_dim=256, time_emb_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.time_emb_dim = time_emb_dim
        
        # Time embedding (sinusoidal)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Main network
        self.input_proj = nn.Linear(latent_dim, hidden_dim)
        
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.SiLU()
            ) for _ in range(3)
        ])
        
        self.output_proj = nn.Linear(hidden_dim, latent_dim)
        
        # Initialize output projection to zero (following DDPM)
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)
    
    def get_time_embedding(self, t, max_period=10000):
        """Create sinusoidal time embeddings."""
        half_dim = self.time_emb_dim // 2
        freqs = torch.exp(
            -torch.log(torch.tensor(max_period)) * torch.arange(half_dim, dtype=torch.float32) / half_dim
        ).to(t.device)
        args = t[:, None].float() * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return embedding
    
    def forward(self, x, t):
        """
        Predict noise given noisy latent x and timestep t.
        
        Args:
            x: Noisy latent of shape (batch, latent_dim)
            t: Timestep of shape (batch,)
        
        Returns:
            Predicted noise of shape (batch, latent_dim)
        """
        # Time embedding
        t_emb = self.get_time_embedding(t)
        t_emb = self.time_mlp(t_emb)
        
        # Input projection
        h = self.input_proj(x)
        
        # Process through blocks with time conditioning
        for block in self.blocks:
            h_block = block(h)
            h = h + h_block + t_emb  # Residual connection with time conditioning
        
        # Output projection
        noise_pred = self.output_proj(h)
        
        return noise_pred

# Initialize diffusion model
diffusion_model = LatentDiffusionModel(latent_dim=latent_dim, hidden_dim=256)
diffusion_model = diffusion_model.to(device)

print(f"Diffusion model initialized")
print(f"Number of parameters: {sum(p.numel() for p in diffusion_model.parameters()):,}")

## 4. Train the Diffusion Model

In [ ]:
def train_diffusion_model(model, latent_codes, noise_schedule, n_epochs=200, batch_size=128, lr=1e-4, device='cpu'):
    """
    Train the diffusion model on latent codes.
    
    Args:
        model: LatentDiffusionModel
        latent_codes: Tensor of shape (n_samples, latent_dim)
        noise_schedule: NoiseSchedule object
        n_epochs: Number of training epochs
        batch_size: Batch size
        lr: Learning rate
        device: Device to train on
    
    Returns:
        Dictionary with training history
    """
    model = model.to(device)
    model.train()
    
    # Create dataloader
    dataset = TensorDataset(latent_codes)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    history = {'loss': []}
    best_loss = float('inf')
    
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0
        
        for (batch_latents,) in dataloader:
            batch_latents = batch_latents.to(device)
            batch_size_actual = batch_latents.shape[0]
            
            # Sample random timesteps
            t = torch.randint(0, noise_schedule.n_steps, (batch_size_actual,), device=device)
            
            # Sample noise
            noise = torch.randn_like(batch_latents)
            
            # Create noisy latents
            noisy_latents = noise_schedule.q_sample(batch_latents, t, noise)
            
            # Predict noise
            noise_pred = model(noisy_latents, t)
            
            # Compute loss (simple MSE)
            loss = F.mse_loss(noise_pred, noise)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        epoch_loss /= n_batches
        history['loss'].append(epoch_loss)
        
        # Update learning rate
        scheduler.step()
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] - Loss: {epoch_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        
        # Save best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'noise_schedule': noise_schedule,
                'latent_dim': latent_dim,
                'epoch': epoch,
                'loss': epoch_loss
            }, 'best_diffusion_model.pth')
    
    return history

# Train the diffusion model
print("\nStarting diffusion model training...")
history = train_diffusion_model(
    diffusion_model,
    latent_codes,
    noise_schedule,
    n_epochs=200,
    batch_size=128,
    lr=1e-4,
    device=device
)

## 5. Visualize Training Progress

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['loss'], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Diffusion Model Training Progress')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 6. Test Sampling from the Diffusion Model

In [ ]:
@torch.no_grad()
def sample_from_diffusion(model, noise_schedule, n_samples=1, latent_dim=64, device='cpu'):
    """
    Sample new latent codes from the diffusion model.
    
    Args:
        model: Trained LatentDiffusionModel
        noise_schedule: NoiseSchedule object
        n_samples: Number of samples to generate
        latent_dim: Dimension of latent space
        device: Device to run on
    
    Returns:
        Generated latent codes of shape (n_samples, latent_dim)
    """
    model.eval()
    
    # Start from pure noise
    x_t = torch.randn(n_samples, latent_dim, device=device)
    
    # Reverse diffusion process
    for t in reversed(range(noise_schedule.n_steps)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        
        # Predict noise
        noise_pred = model(x_t, t_batch)
        
        # Compute denoising step
        alpha_t = noise_schedule.alphas[t]
        alpha_cumprod_t = noise_schedule.alphas_cumprod[t]
        beta_t = noise_schedule.betas[t]
        
        # Predict x_0
        x_0_pred = (x_t - torch.sqrt(1 - alpha_cumprod_t) * noise_pred) / torch.sqrt(alpha_cumprod_t)
        
        # Compute x_{t-1}
        if t > 0:
            noise = torch.randn_like(x_t)
            alpha_cumprod_t_prev = noise_schedule.alphas_cumprod[t - 1]
            
            # DDPM formula
            x_t = (
                torch.sqrt(alpha_cumprod_t_prev) * beta_t / (1 - alpha_cumprod_t) * x_0_pred +
                torch.sqrt(alpha_t) * (1 - alpha_cumprod_t_prev) / (1 - alpha_cumprod_t) * x_t +
                torch.sqrt(noise_schedule.posterior_variance[t]) * noise
            )
        else:
            x_t = x_0_pred
    
    return x_t

# Load best model
checkpoint = torch.load('best_diffusion_model.pth', map_location=device)
diffusion_model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded (epoch {checkpoint['epoch']}, loss {checkpoint['loss']:.6f})")

# Generate some samples
print("\nGenerating latent samples from diffusion model...")
n_samples = 100
generated_latents = sample_from_diffusion(
    diffusion_model,
    noise_schedule,
    n_samples=n_samples,
    latent_dim=latent_dim,
    device=device
).cpu()

print(f"Generated {n_samples} latent codes")
print(f"Shape: {generated_latents.shape}")

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 2D projection comparison
axes[0].scatter(latent_codes[:, 0].numpy(), latent_codes[:, 1].numpy(), 
                alpha=0.3, s=10, label='Real latents', c='blue')
axes[0].scatter(generated_latents[:, 0].numpy(), generated_latents[:, 1].numpy(), 
                alpha=0.5, s=20, label='Generated latents', c='red', marker='x')
axes[0].set_xlabel('Latent Dimension 1')
axes[0].set_ylabel('Latent Dimension 2')
axes[0].set_title('Real vs Generated Latent Codes (2D projection)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution comparison
axes[1].hist(latent_codes.numpy().flatten(), bins=50, alpha=0.5, 
             label='Real', density=True, edgecolor='black')
axes[1].hist(generated_latents.numpy().flatten(), bins=50, alpha=0.5, 
             label='Generated', density=True, edgecolor='black')
axes[1].set_xlabel('Latent Value')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save the Trained Diffusion Model

In [ ]:
# Save the complete diffusion model
os.makedirs('models', exist_ok=True)

torch.save({
    'model_state_dict': diffusion_model.state_dict(),
    'noise_schedule_params': {
        'n_steps': noise_schedule.n_steps,
        'beta_start': noise_schedule.betas[0].item(),
        'beta_end': noise_schedule.betas[-1].item()
    },
    'latent_dim': latent_dim,
    'hidden_dim': 256,
    'time_emb_dim': 64
}, 'models/latent_diffusion_model.pth')

print("Diffusion model saved successfully!")
print("  - models/latent_diffusion_model.pth")

## Summary

In this notebook, we:

1. ✅ Loaded the pre-trained **Encoder** from Phase 1
2. ✅ Encoded functional training data into latent space
3. ✅ Implemented a **Diffusion Model (DDPM)** for the latent space
4. ✅ Trained the diffusion model to learn the latent distribution
5. ✅ Tested sampling new latent codes from noise
6. ✅ Saved the trained diffusion model

### Next Steps

Proceed to `03_functional_generation_demo.ipynb` to:
- Load both the **Encoder**, **Decoder**, and **Diffusion Model**
- Generate complete functional data samples:
  1. Sample latent code `z` from diffusion model
  2. Decode `z` to function `x(t)` at arbitrary discretization
- Visualize and validate synthetic functional data